# ARC_ATLAS v4 (self-contained)

End-to-end training notebook that only depends on:
- raw ARC + ATLAS data outside this folder (see `config/paths.yaml`)
- everything else lives inside this folder after you run the prep step.

Steps:
1. (Optional) Materialize the processed split locally (copies, no symlinks).
2. Train SmartSOTA dynamic model on hires split.
3. (Optional) Resume from a prior run.
4. (Optional) Quick sanity predictions.


In [1]:
from pathlib import Path
import importlib.util
import shutil
import time
import traceback

# --------- Paths and module loading ---------
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
RUN_ROOT = PROJECT_ROOT
SRC = PROJECT_ROOT / "src" / "training_v2.py"

TRAIN_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/80_20_random/train")
TRAIN_T1 = TRAIN_DIR / "t1"
TRAIN_MASKS = TRAIN_DIR / "masks"

if not SRC.exists():
    raise FileNotFoundError(f"Training module not found: {SRC}")
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"Training data dir not found: {TRAIN_DIR}. Run ARC_ATLAS_TrainPrep_v4.ipynb first.")
if not TRAIN_T1.exists() or not TRAIN_MASKS.exists():
    raise FileNotFoundError(f"Expected subfolders missing under {TRAIN_DIR}: t1/ and masks/")

spec = importlib.util.spec_from_file_location("seg", SRC)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load module spec from {SRC}")
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

# --------- Hyperparameters ---------
INPUT_SHAPE = (112, 112, 96, 1)
PATCH_SIZE = (112, 112, 96)
PATCHES_PER_CASE = 2
EPOCH_STEPS = 2000
FIT_VERBOSE = 2
MEMORY_LOGS_ENABLED = False
TOTAL_EPOCHS = 200
INITIAL_EPOCH = 0

BASE_FILTERS = 6
SAM_HEADS = 2
BATCH_SIZE = 2
VAL_SPLIT = 0.15
DROPOUT_RATE = 0.55
L2_REG = 0.0015

AUG_INTENSITY = 0.45
ROTATION_RANGE = 25
SMALL_LESION_THRESHOLD = 6000
SYNTHETIC_LESION_PROB = 0.6

INITIAL_LR = 1e-4
MIN_LR = 5e-7
WARMUP_EPOCHS = 15
COSINE_FIRST_CYCLE_EPOCHS = 140
COSINE_T_MUL = 1.0
COSINE_M_MUL = 1.0
SWA_EPOCHS = 0
SWA_LR_MULT = None

DICE_WEIGHT = 0.4
BOUNDARY_WEIGHT = 0.6
BOUNDARY_WARMUP_DICE = 0.4
BOUNDARY_WARMUP_BOUNDARY = 0.6
BOUNDARY_RAMP_EPOCHS = 1

FOCAL_TVERSKY_WEIGHT = 0.0
TVERSKY_ALPHA = 0.7
TVERSKY_BETA = 0.3
FOCAL_TVERSKY_GAMMA = 1.5

SIZE_BUCKET_PROBS = (0.35, 0.25, 0.20, 0.12, 0.08)
PATCH_FG_PROB_BY_BIN = (0.95, 0.90, 0.80, 0.65, 0.55)

# Full-image patch extraction controls
LOAD_FULL_IMAGE_FOR_PATCHING = True
FULL_RES_TARGET_SHAPE = None
WHOLE_BRAIN_VAL_ENABLED = True
WHOLE_BRAIN_VAL_EVERY_N_EPOCHS = 1
WHOLE_BRAIN_VAL_MAX_CASES = None
WHOLE_BRAIN_VAL_TTA = False
PATCH_SAMPLING_STRATEGY = "hemisphere"
HEMISPHERE_AXIS = 2
HEMISPHERE_BALANCED = True

# --------- Per-run artifact directories ---------
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
for d in (MODEL_DIR, CALLBACKS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Using training module:", SRC)
print("Training data:", TRAIN_DIR)
print("Run dir:", RUN_DIR)

# --------- Train fresh run ---------
try:
    history = seg.train_dynamic_model(
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,
        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,
        INPUT_SHAPE=INPUT_SHAPE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        BATCH_SIZE=BATCH_SIZE,
        DROPOUT_RATE=DROPOUT_RATE,
        L2_REG=L2_REG,
        PATCH_SIZE=PATCH_SIZE,
        PATCHES_PER_CASE=PATCHES_PER_CASE,
        EPOCH_STEPS=EPOCH_STEPS,
        FIT_VERBOSE=FIT_VERBOSE,
        MEMORY_LOGS_ENABLED=MEMORY_LOGS_ENABLED,
        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        RESAMPLE_TO_TARGET=False,
        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        ROTATION_RANGE=ROTATION_RANGE,
        SMALL_LESION_THRESHOLD=SMALL_LESION_THRESHOLD,
        SYNTHETIC_LESION_PROB=SYNTHETIC_LESION_PROB,
        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,
        COSINE_FIRST_CYCLE_EPOCHS=COSINE_FIRST_CYCLE_EPOCHS,
        COSINE_T_MUL=COSINE_T_MUL,
        COSINE_M_MUL=COSINE_M_MUL,
        COSINE_MIN_LR_MULT=0.1,
        SWA_EPOCHS=SWA_EPOCHS,
        SWA_LR_MULT=SWA_LR_MULT,
        DICE_WEIGHT=DICE_WEIGHT,
        BOUNDARY_WEIGHT=BOUNDARY_WEIGHT,
        DICE_LOSS_WEIGHT=0.4,
        BOUNDARY_LOSS_WEIGHT=0.6,
        BOUNDARY_WARMUP_DICE=BOUNDARY_WARMUP_DICE,
        BOUNDARY_WARMUP_BOUNDARY=BOUNDARY_WARMUP_BOUNDARY,
        BOUNDARY_RAMP_EPOCHS=BOUNDARY_RAMP_EPOCHS,
        FOCAL_TVERSKY_WEIGHT=FOCAL_TVERSKY_WEIGHT,
        TVERSKY_ALPHA=TVERSKY_ALPHA,
        TVERSKY_BETA=TVERSKY_BETA,
        FOCAL_TVERSKY_GAMMA=FOCAL_TVERSKY_GAMMA,
        SIZE_BUCKET_PROBS=SIZE_BUCKET_PROBS,
        PATCH_FG_PROB_BY_BIN=PATCH_FG_PROB_BY_BIN,
        LOAD_FULL_IMAGE_FOR_PATCHING=LOAD_FULL_IMAGE_FOR_PATCHING,
        FULL_RES_TARGET_SHAPE=FULL_RES_TARGET_SHAPE,
        WHOLE_BRAIN_VAL_ENABLED=WHOLE_BRAIN_VAL_ENABLED,
        WHOLE_BRAIN_VAL_EVERY_N_EPOCHS=WHOLE_BRAIN_VAL_EVERY_N_EPOCHS,
        WHOLE_BRAIN_VAL_MAX_CASES=WHOLE_BRAIN_VAL_MAX_CASES,
        WHOLE_BRAIN_VAL_TTA=WHOLE_BRAIN_VAL_TTA,
        PATCH_SAMPLING_STRATEGY=PATCH_SAMPLING_STRATEGY,
        HEMISPHERE_AXIS=HEMISPHERE_AXIS,
        HEMISPHERE_BALANCED=HEMISPHERE_BALANCED,
        DIFF_AWARE_ENABLED=True,
        DIFF_EMA_LAMBDA=0.8,
        DIFF_BETA=1.5,
        VALIDATION_SPLIT=VAL_SPLIT,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,
    )
    print("Training complete. Keys:", list(getattr(history, "history", {}).keys()))
    print("Artifacts saved to", RUN_DIR)
except Exception:
    traceback.print_exc()
    raise

# Convenience: mark this run as latest
latest_link = RUN_ROOT / "runs" / "latest"
if latest_link.exists() or latest_link.is_symlink():
    latest_link.unlink()
latest_link.symlink_to(RUN_DIR, target_is_directory=True)

best_src = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if best_src.exists():
    best_copy = RUN_ROOT / "runs" / "latest_best.weights.h5"
    shutil.copy2(best_src, best_copy)
    print("Saved best copy ->", best_copy)



2026-03-05 17:04:41.929534: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1772755484.055383 3946970 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1772755484.056456 3946970 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1772755484.056798 3946970 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1772755484.057787 3946970 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2026-03-05 17:04:44,123 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-03-05 17:04:44,124 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-03-05 17:04:44,124 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy
Using training module: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/src/training_v2.py
Training data: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/80_20_random/train
Run dir: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260305_170444


2026-03-05 17:04:45,361 - SmartSOTA_Dynamic - INFO - Model built: 1,568,455 parameters
2026-03-05 17:04:45,361 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2026-03-05 17:04:45,362 - SmartSOTA_Dynamic - INFO - 📄 Using manifest-defined pairs from /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/80_20_random/train/manifest.csv
2026-03-05 17:06:18,962 - SmartSOTA_Dynamic - INFO - Manifest composition: {'ARC-t1w-standardized-5893ef9b': 165, 'ATLAS-Images-f0d7431e': 518, 'Approx-Numeracy-Processed': 87}
2026-03-05 17:06:18,963 - SmartSOTA_Dynamic - INFO - 📊 Created 770 image–mask pairs from manifest
2026-03-05 17:06:18,963 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 99.61%
2026-03-05 17:08:24,673 - SmartSOTA_Dynamic - WARNING - create_stratified_splits not found; using simple random split fallback.
2026-03-05 17:08:24,698 - SmartSOTA_Dynamic - WARNING - NVMLMemoryLogger not available; GPU telemetry callback disabled.


...

2026-03-06 11:59:28,039 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 8/116 cases
2026-03-06 12:00:55,247 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 16/116 cases
2026-03-06 12:02:22,613 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 24/116 cases
2026-03-06 12:03:49,728 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 32/116 cases
2026-03-06 12:05:16,604 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 40/116 cases
2026-03-06 12:06:43,470 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 48/116 cases
2026-03-06 12:08:10,056 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 56/116 cases
2026-03-06 12:09:37,148 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 64/116 cases
2026-03-06 12:11:04,109 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 72/116 cases
2026-03-06 12:12:31,302 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 80/116 cases
2026-03-06 12:13:58,512 - SmartSOTA_Dynamic - INFO - Whole-brain val progress: 88


Epoch 18: val_dice_coefficient did not improve from 0.05927
2000/2000 - 3791s - 2s/step - dice_coefficient: 0.1533 - loss: 0.7029 - safe_binary_iou: 0.0922 - val_dice_coefficient: 0.0199 - val_whole_dice_micro: 0.0357 - val_whole_dice_hard: 0.0023


2026-03-06 12:19:03,330 - SmartSOTA_Dynamic - INFO - 📉 Loss mix @epoch 18: dice=0.400, boundary=0.600, focal=0.000


Epoch 19/200


KeyboardInterrupt: 

In [ ]:
# --------- Quick sanity prediction on zeros ---------
import numpy as np

cfg = seg.DynamicTrainingConfig(
    DATA_DIR=TRAIN_DIR,
    IMAGES_DIR=TRAIN_T1,
    MASKS_DIR=TRAIN_MASKS,
    INPUT_SHAPE=INPUT_SHAPE,
    PATCH_SIZE=PATCH_SIZE,
    MODEL_DIR=MODEL_DIR,
    CALLBACKS_DIR=CALLBACKS_DIR,
)

weights = CALLBACKS_DIR / "best_model_dynamic.weights.h5"
if weights.exists():
    m = seg.build_model_for_inference(cfg, weights_path=str(weights))
else:
    m = seg.build_model_for_inference(cfg)

x0 = np.zeros((1, *INPUT_SHAPE), np.float32)
p0 = m.predict(x0, verbose=0)[0, ..., 0]
print("Blank input -> p.mean=", float(p0.mean()), " p.max=", float(p0.max()))


Blank input -> p.mean= 0.10394287109375  p.max= 0.95703125
